# Screening Decisions Analysis

Analyze the paper screening decisions made by each rater and export the final included papers as a BibTeX file.

In [ ]:
import re
from pathlib import Path

import bibtexparser
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").exists()
)
LUCAS_DECISIONS_PATH = PROJECT_ROOT / "decisions" / "lucas.csv"
FIRST_DECISIONS_PATH = PROJECT_ROOT / "decisions" / "first.csv"
ALL_PAPERS_PATH = PROJECT_ROOT / "artifacts" / "all_papers.csv"
LIBRARY_PATH = PROJECT_ROOT / "papers" / "papers.bib"
INCLUDED_BIB_PATH = PROJECT_ROOT / "artifacts" / "included.bib"

## Screening Decisions

Load each rater's decisions and inspect the include/exclude distribution.

In [2]:
lucas_df = pd.read_csv(LUCAS_DECISIONS_PATH, usecols=["id", "decision", "reason"])
lucas_df.head()

,id,decision,reason
0,52,include,IC1
1,60,include,IC2
2,71,include,IC2
3,77,include,IC1
4,107,exclude,EC2


In [3]:
lucas_df["decision"].value_counts()

decision
exclude    210
include     34
Name: count, dtype: int64

In [4]:
first_df = pd.read_csv(FIRST_DECISIONS_PATH)
first_df.head()

,id,decision,reason
0,1,include,IC1
1,2,include,IC1
2,3,include,IC1
3,4,include,IC1
4,5,include,IC1


In [25]:
all_decisions = pd.concat([first_df, lucas_df])
all_decisions["decision"].value_counts()

decision
exclude    545
include     64
Name: count, dtype: int64

In [14]:
to_include_df = lucas_df[lucas_df["decision"] == "include"]
to_include_df.head()

,id,decision,reason
0,52,include,IC1
1,60,include,IC2
2,71,include,IC2
3,77,include,IC1
13,188,include,IC1


## BibTeX Export

Load the full BibTeX library and `artifacts/all_papers.csv`, match each included paper by title, and write the results to `artifacts/included.bib`.

In [ ]:
with LIBRARY_PATH.open(encoding="utf-8") as f:
    library = bibtexparser.parse_string(f.read())

In [23]:
papers_df = pd.read_csv(ALL_PAPERS_PATH, usecols=["id", "canonical_id", "title", "abstract"])
included_papers = papers_df[papers_df["id"].isin(to_include_df["id"])]
print(f"Papers to include: {len(included_papers)}")
included_papers.head()

Papers to include: 34


,id,canonical_id,title,abstract
51,52,52,The Power of\&nbsp;Training: How Different Neu...,This work offers a heuristic evaluation of the...
59,60,60,The Energy Efficiency Research of\&nbsp;Code f...,Last ten years the top performance of the fast...
70,71,71,What A Waste,The immense demand for high performance comput...
76,77,77,Adaptive Carbon-Aware Scheduling Policies for\...,In response to growing energy costs and carbon...
187,188,188,Enabling distributed generation powered sustai...,The necessity for capping carbon emission has ...


In [21]:
import re

def normalize_title(t):
    # Collapse \cmd{arg} -> \cmdarg so CSV and bib titles compare equal
    return re.sub(r'\{(\w+)\}', r'\1', t).strip()

title_to_entry = {
    normalize_title(e.fields_dict["title"].value): e
    for e in library.entries
    if "title" in e.fields_dict
}

included_entries = []
missing = []
for _, row in included_papers.iterrows():
    entry = title_to_entry.get(normalize_title(row["title"]))
    if entry:
        included_entries.append(entry)
    else:
        missing.append(row["id"])

print(f"Found: {len(included_entries)}, Missing: {len(missing)}")
if missing:
    print("Missing IDs:", missing)

Found: 34, Missing: 0


In [22]:
out_lib = bibtexparser.Library()
for entry in included_entries:
    out_lib.add(entry)

bibtex_str = bibtexparser.write_string(out_lib)
with INCLUDED_BIB_PATH.open("w", encoding="utf-8") as f:
    f.write(bibtex_str)

print(f"Written {len(included_entries)} entries to {INCLUDED_BIB_PATH.relative_to(PROJECT_ROOT)}")

Written 34 entries to artifacts/included.bib
